In [5]:
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/arjunmahesh09999/new-masterdata/MASTERDATA.csv


In [6]:
df = pd.read_csv("/kaggle/input/datasets/arjunmahesh09999/new-masterdata/MASTERDATA.csv")
print(f"Loaded: {df.shape[0]:,} rows | {df['patient_id'].nunique()} patients")

Loaded: 2,378,857 rows | 386 patients


In [7]:
# ── USER INPUT ──────────────────────────────────────────
TARGET_PATIENT_ID = 64   # Patient ID to inspect
TARGET_TIME       = 3600  # Time in seconds (3600 = 60 min)
# ────────────────────────────────────────────────────────

In [8]:
# Extract current and 15-min-ago (lag) snapshots
current_stat = df[
    (df['patient_id'] == TARGET_PATIENT_ID) & (df['time'] == TARGET_TIME)
].reset_index(drop=True)

lag_stat = df[
    (df['patient_id'] == TARGET_PATIENT_ID) & (df['time'] == TARGET_TIME - 900)
].reset_index(drop=True)

if current_stat.empty:
    raise ValueError(f"No data for Patient {TARGET_PATIENT_ID} at time {TARGET_TIME}s.")

past_stat = None if lag_stat.empty else lag_stat
print(f"Snapshot ready — Patient {TARGET_PATIENT_ID} @ {TARGET_TIME}s")
print(f"Lag snapshot (15m ago): {'Found' if past_stat is not None else 'Not available'}")

Snapshot ready — Patient 64 @ 3600s
Lag snapshot (15m ago): Found


In [9]:
# ── THRESHOLDS  (Normal, Critical, Emergency) ───────────
TH = {
    "spo2":       (95,  92,  90),
    "hr_low":     (60,  50,  45),   "hr_high":    (90,  110, 120),
    "rr_low":     (12,  10,   8),   "rr_high":    (20,   25,  30),
    "sbp_low":    (110, 100,  90),  "sbp_high":   (150, 170, 185),
    "dbp_low":    (60,   55,  50),  "dbp_high":   (85,   95, 100),
    "mbp":        (70,   65,  60),
    "etco2_low":  (35,   30,  25),  "etco2_high": (45,   50,  55),
    "pp_low":     (45,   35,  30),  "pp_high":    (65,   75,  85),
}

# ── VITAL DIRECTION MAP ──────────────────────────────────
# Explicitly declares which directions are valid for each vital.
# Vitals with only a bare key in TH (e.g. 'spo2', 'mbp') must
# have their allowed alert directions declared here to prevent
# the fallback-key logic from firing spurious HIGH or LOW alerts.
#
# 'low'  → only flag when value is BELOW threshold  (e.g. SpO₂, MAP)
# 'high' → only flag when value is ABOVE threshold  (e.g. a high-only vital)
# 'both' → flag in either direction (default for vitals with _low/_high keys)
VITAL_DIRECTION = {
    "spo2": "low",   # SpO₂ 100% is healthy — only low values are dangerous
    "mbp":  "low",   # MAP threshold in TH is the LOWER bound (normal ≥ 70 mmHg)
}
# Vitals not listed here default to 'both' (they have explicit _low / _high TH keys)


# ── CONDITION FACTOR SPECS ───────────────────────────────
CONDITION_FACTOR_SPECS = {
    "t1_shock_spiral": [
        {"raw": "mbp",               "dir": "low",  "thresh": 70,  "normal_ref": 90},
        {"raw": "heart_rate",        "dir": "high", "thresh": 100, "normal_ref": 75},
    ],
    "t1_resp_burnout": [
        {"raw": "spo2",              "dir": "low",  "thresh": 92,  "normal_ref": 98},
        {"raw": "resp_rate_smoothed","dir": "high", "thresh": 22,  "normal_ref": 16},
    ],
    "t1_hypercapnic": [
        {"raw": "etco2",             "dir": "high", "thresh": 50,  "normal_ref": 40},
        {"raw": "resp_rate_smoothed","dir": "low",  "thresh": 10,  "normal_ref": 16},
    ],
    "t2_pulse_pressure_low": [
        {"raw": "pulse_pressure",    "dir": "low",  "thresh": 30,  "normal_ref": 50},
    ],
    "t2_widepp_highsbp": [
        {"raw": "pulse_pressure",    "dir": "high", "thresh": 70,  "normal_ref": 50},
        {"raw": "sbp",               "dir": "high", "thresh": 170, "normal_ref": 120},
    ],
    "t2_resp_hemo_combo": [
        {"raw": "spo2",              "dir": "low",  "thresh": 92,  "normal_ref": 98},
        {"raw": "resp_rate_smoothed","dir": "high", "thresh": 22,  "normal_ref": 16},
        {"raw": "heart_rate",        "dir": "high", "thresh": 100, "normal_ref": 75},
    ],
    "t3_hyper_emergency": [
        {"raw": "sbp",               "dir": "high", "thresh": 180, "normal_ref": 120},
        {"raw": "pulse_pressure",    "dir": "high", "thresh": 70,  "normal_ref": 50},
    ],
    "t3_stable_deceiver": [
        {"raw": "spo2",              "dir": "low",  "thresh": 94,  "normal_ref": 98},
        {"raw": "heart_rate",        "dir": "low",  "thresh": 90,  "normal_ref": 100},
        {"raw": "mbp",               "dir": "low",  "thresh": 70,  "normal_ref": 90},
    ],
    "t3_masked_shock": [
        {"raw": "mbp",               "dir": "low",  "thresh": 72,  "normal_ref": 90},
        {"raw": "heart_rate",        "dir": "low",  "thresh": 90,  "normal_ref": 100},
    ],
    "t3_occult_acidosis": [
        {"raw": "etco2",             "dir": "low",  "thresh": 32,  "normal_ref": 40},
        {"raw": "resp_rate_smoothed","dir": "high", "thresh": 24,  "normal_ref": 16},
        {"raw": "spo2",              "dir": "low",  "thresh": 92,  "normal_ref": 98},
    ],
    "t3_trend_decline": [
        {"raw": "etco2",             "dir": "high", "thresh": 50,  "normal_ref": 40},
        {"raw": "spo2",              "dir": "low",  "thresh": 92,  "normal_ref": 98},
        {"raw": "heart_rate",        "dir": "high", "thresh": 100, "normal_ref": 75},
    ],
    "t3_trend_activate": [
        {"raw": "spo2",              "dir": "low",  "thresh": 94,  "normal_ref": 98},
        {"raw": "heart_rate",        "dir": "high", "thresh": 95,  "normal_ref": 75},
        {"raw": "etco2",             "dir": "low",  "thresh": 35,  "normal_ref": 40},
    ],
}

CONDITIONS = {k: {"tier": int(k[1])} for k in CONDITION_FACTOR_SPECS}

VITALS_MAP = {
    "spo2":               "spo2",
    "heart_rate":         "hr",
    "resp_rate_smoothed": "rr",
    "sbp":                "sbp",
    "dbp":                "dbp",
    "mbp":                "mbp",
    "etco2":              "etco2",
    "pulse_pressure":     "pp",
}

# Human-friendly vital names for narrative output
VITAL_NAMES = {
    "spo2":               "Oxygen Saturation (SpO₂)",
    "heart_rate":         "Heart Rate",
    "resp_rate_smoothed": "Respiratory Rate",
    "sbp":                "Systolic Blood Pressure (SBP)",
    "dbp":                "Diastolic Blood Pressure (DBP)",
    "mbp":                "Mean Arterial Pressure (MAP)",
    "etco2":              "End-Tidal CO₂ (EtCO₂)",
    "pulse_pressure":     "Pulse Pressure",
}

VITAL_UNITS = {
    "spo2":               "%",
    "heart_rate":         "bpm",
    "resp_rate_smoothed": "breaths/min",
    "sbp":                "mmHg",
    "dbp":                "mmHg",
    "mbp":                "mmHg",
    "etco2":              "mmHg",
    "pulse_pressure":     "mmHg",
}

# Clinical meaning of each vital being high or low
VITAL_CLINICAL_MEANING = {
    "spo2":               {"low":  "indicates the patient's blood oxygen level is dropping — tissues may not be receiving enough oxygen",
                           "high": "is within safe range (hyperoxia rarely flagged by this system)"},
    "heart_rate":         {"high": "indicates the heart is working harder than normal, possibly compensating for low blood pressure, pain, fever, or fluid loss",
                           "low":  "indicates the heart rate is dangerously slow, which may reduce cardiac output and organ perfusion"},
    "resp_rate_smoothed": {"high": "means the patient is breathing faster than normal — the body may be trying to compensate for low oxygen or acidosis",
                           "low":  "means breathing is dangerously slow, which can lead to CO₂ retention and respiratory failure"},
    "sbp":                {"high": "indicates elevated systolic pressure — hypertensive crisis risk",
                           "low":  "indicates hypotension — the heart may not be pumping enough blood to vital organs"},
    "dbp":                {"high": "indicates elevated diastolic pressure",
                           "low":  "indicates low diastolic pressure, potentially reducing coronary perfusion"},
    "mbp":                {"low":  "is critically important — MAP below 70 mmHg is associated with organ ischemia and shock",
                           "high": "indicates elevated mean arterial pressure"},
    "etco2":              {"high": "suggests CO₂ is building up — hypoventilation or metabolic acidosis compensation",
                           "low":  "suggests hyperventilation or poor cardiac output with reduced CO₂ delivery to the lungs"},
    "pulse_pressure":     {"low":  "is a critical warning sign of tamponade, severe hypovolemia, or cardiogenic shock — the difference between systolic and diastolic is narrowing dangerously",
                           "high": "suggests aortic regurgitation, high systolic with low diastolic, or vascular stiffness"},
}

# Condition readable names and clinical descriptions
CONDITION_NAMES = {
    "t1_shock_spiral":      "Shock Spiral (Tier 1)",
    "t1_resp_burnout":      "Respiratory Burnout (Tier 1)",
    "t1_hypercapnic":       "Hypercapnic Respiratory Failure (Tier 1)",
    "t2_pulse_pressure_low":"Critically Low Pulse Pressure (Tier 2)",
    "t2_widepp_highsbp":    "Wide Pulse Pressure + Hypertension (Tier 2)",
    "t2_resp_hemo_combo":   "Respiratory-Hemodynamic Combination (Tier 2)",
    "t3_hyper_emergency":   "Hypertensive Emergency (Tier 3)",
    "t3_stable_deceiver":   "Stable Deceiver Pattern (Tier 3)",
    "t3_masked_shock":      "Masked Shock (Tier 3)",
    "t3_occult_acidosis":   "Occult Acidosis (Tier 3)",
    "t3_trend_decline":     "Trend Decline Pattern (Tier 3)",
    "t3_trend_activate":    "Trend Activate Pattern (Tier 3)",
}

CONDITION_CLINICAL_DESC = {
    "t1_shock_spiral":      "The patient's MAP is falling while the heart rate is rising — the body is compensating for poor perfusion. This is the hallmark early pattern of circulatory shock.",
    "t1_resp_burnout":      "SpO₂ is low while respiratory rate is elevated — the lungs are working at maximum effort but failing to maintain oxygenation. This indicates respiratory muscle fatigue may be near.",
    "t1_hypercapnic":       "EtCO₂ is high but respiratory rate is low — the patient is retaining CO₂ due to insufficient ventilation. This can rapidly progress to respiratory acidosis.",
    "t2_pulse_pressure_low":"The pulse pressure is critically narrow, suggesting the heart's stroke volume is dropping. This is a red flag for tamponade, severe hypovolemia, or cardiogenic shock.",
    "t2_widepp_highsbp":    "Wide pulse pressure combined with very high systolic BP — this pattern is associated with aortic regurgitation or severe hypertensive states.",
    "t2_resp_hemo_combo":   "A combined respiratory and hemodynamic failure pattern — low SpO₂, fast breathing, and elevated heart rate are all occurring simultaneously, indicating the patient is under significant physiological stress.",
    "t3_hyper_emergency":   "Extreme systolic hypertension with wide pulse pressure — this is a hypertensive emergency with high risk of end-organ damage including stroke, cardiac failure, or aortic dissection.",
    "t3_stable_deceiver":   "Vitals appear borderline normal individually, but together they form a pattern of compensated deterioration. The patient may look stable but the body is masking a deeper problem.",
    "t3_masked_shock":      "MAP and heart rate together suggest a shock state that may not yet be obvious clinically. The system is detecting the early biochemical footprint of masked circulatory failure.",
    "t3_occult_acidosis":   "Low EtCO₂, fast respiratory rate, and falling SpO₂ together suggest an occult acidosis state — the body is hyperventilating to compensate for metabolic acid buildup.",
    "t3_trend_decline":     "A trend-based deterioration pattern — EtCO₂ is rising, SpO₂ is falling, and heart rate is climbing. Even if individual values are borderline, the direction is toward crisis.",
    "t3_trend_activate":    "An early activation pattern — SpO₂ is slipping, heart rate is rising, and EtCO₂ is falling. This patient's trajectory needs immediate attention before it escalates.",
}

print("Thresholds, condition specs, and clinical dictionaries loaded.")

Thresholds, condition specs, and clinical dictionaries loaded.


In [10]:
# ── STAGE 1 & 2: System Labels + Mismatch Explanation ───

def stage_1_2(row):
    curr  = row['severity_label'].values[0]   # 0=Normal, 1=Critical, 2=Emergency
    conf  = row['result_label'].values[0]
    score = row['combined_score'].values[0]
    LABEL = {0: "Normal", 1: "Critical", 2: "Emergency"}

    print("═" * 65)
    print("     RULE-BASED EXPLAINABLE AI LAYER — PATIENT VITAL ANALYSIS")
    print("═" * 65)
    print(f"  Patient ID      : {TARGET_PATIENT_ID}")
    print(f"  Time Point      : {TARGET_TIME}s  ({TARGET_TIME // 60} min into monitoring)")
    print(f"  Combined Score  : {score:.4f}")
    print("─" * 65)

    # Stage 1: label display
    print("  STAGE 1 — SYSTEM LABELS")
    print(f"  Patient Current Condition  : {LABEL.get(curr, curr)}   (severity_label)")
    print(f"  System Confirmed Condition : {LABEL.get(conf, conf)}   (result_label)")
    print("─" * 65)

    # Stage 2: mismatch explanation
    print("  STAGE 2 — CONDITION MISMATCH EXPLANATION")

    if curr != conf:
        # Case A: current is Normal but confirmed is not
        if curr == 0 and conf != 0:
            print(
                f"\n  👉 The patient's vitals appear Normal at this exact moment."
                f"\n     However, the system has not yet confirmed a return to Normal."
                f"\n     This is because the FSM (Finite State Machine) evaluates vitals"
                f"\n     across a 15-reading continuous window before changing its confirmed label."
                f"\n"
                f"\n     📋 Previous vital readings were indicating a {LABEL.get(conf, conf)} state."
                f"\n     The system is currently waiting to confirm whether this recovery is"
                f"\n     sustained or a temporary fluctuation."
                f"\n"
                f"\n     ⏳ Status: Vitals are normal at this stage, but prior vitals indicate"
                f"\n        a deterioration episode. Awaiting system confirmation of Normal"
                f"\n        or a reduction in severity labeling."
            )
        # Case B: current is not Normal, confirmed is different
        else:
            print(
                f"\n  👉 Patient current condition is {LABEL.get(curr, curr)}, and it is being evaluated"
                f"\n     over a 15 continuous vital-checking window using system logic."
                f"\n"
                f"\n     By analysis, the patient is under \"{LABEL.get(conf, conf)}\" condition as confirmed"
                f"\n     by the FSM state machine, which requires sustained readings before"
                f"\n     committing to a new label — this prevents false alarm spikes."
                f"\n"
                f"\n     Based on this sustained assessment, the system confirms the"
                f"\n     patient's condition as: {LABEL.get(conf, conf).upper()}."
            )
    else:
        # Both match — Normal case
        if curr == 0:
            print(
                f"\n  ✅ Vitals are Normal."
                f"\n     Both the raw severity reading and the system-confirmed label agree:"
                f"\n     the patient is in a stable, normal physiological state."
                f"\n     All vitals are within expected ranges and the FSM has confirmed"
                f"\n     this Normal state over the continuous monitoring window."
            )
        # Both match — Critical or Emergency
        else:
            print(
                f"\n  🚨 Both the raw severity and the system-confirmed label are in"
                f"\n     agreement: the patient is in a confirmed {LABEL.get(conf, conf).upper()} state."
                f"\n"
                f"\n     This means the deterioration is not a transient spike —"
                f"\n     it has been sustained across the 15-reading evaluation window"
                f"\n     and the FSM has locked in this severity label."
                f"\n     Immediate clinical attention is required."
            )

    return curr, conf

curr_state, conf_state = stage_1_2(current_stat)

═════════════════════════════════════════════════════════════════
     RULE-BASED EXPLAINABLE AI LAYER — PATIENT VITAL ANALYSIS
═════════════════════════════════════════════════════════════════
  Patient ID      : 64
  Time Point      : 3600s  (60 min into monitoring)
  Combined Score  : 1.2054
─────────────────────────────────────────────────────────────────
  STAGE 1 — SYSTEM LABELS
  Patient Current Condition  : Critical   (severity_label)
  System Confirmed Condition : Critical   (result_label)
─────────────────────────────────────────────────────────────────
  STAGE 2 — CONDITION MISMATCH EXPLANATION

  🚨 Both the raw severity and the system-confirmed label are in
     agreement: the patient is in a confirmed CRITICAL state.

     This means the deterioration is not a transient spike —
     it has been sustained across the 15-reading evaluation window
     and the FSM has locked in this severity label.
     Immediate clinical attention is required.


In [11]:
# ── STAGE 3: Vital Sign Analysis + Pattern Flags ─────────

def _get_level(val, pfx, direction):
    """Return 'Normal', 'Critical', or 'Emergency' for a given vital value."""
    if direction == "high":
        key = f"{pfx}_high" if f"{pfx}_high" in TH else pfx
        if key not in TH:
            return "Normal"
        thresholds = TH[key]
        if val > thresholds[2]:
            return "Emergency"
        elif val > thresholds[1]:
            return "Critical"
        elif val > thresholds[0]:
            return "Borderline-High"
    else:
        key = f"{pfx}_low" if f"{pfx}_low" in TH else pfx
        if key not in TH:
            return "Normal"
        thresholds = TH[key]
        if val < thresholds[2]:
            return "Emergency"
        elif val < thresholds[1]:
            return "Critical"
        elif val < thresholds[0]:
            return "Borderline-Low"
    return "Normal"


def stage_3(row, curr_state, conf_state):
    LABEL = {0: "Normal", 1: "Critical", 2: "Emergency"}

    print("\n" + "═" * 65)
    print("  STAGE 3 — VITAL SIGN ANALYSIS & CLINICAL PATTERN FLAGS")
    print("═" * 65)

    # ── Normal cases — exit early ──────────────────────────
    if curr_state == 0:
        if conf_state != 0:
            print(
                f"\n  👉 Vitals are within normal range at this moment."
                f"\n     However, prior vital readings were indicating a deterioration"
                f"\n     toward {LABEL.get(conf_state, conf_state)} status."
                f"\n"
                f"\n     The system is currently in a watchful holding state — waiting"
                f"\n     to confirm whether the patient is genuinely recovering or whether"
                f"\n     this is a brief stable interval before further decline."
                f"\n"
                f"\n     🔔 Continue close monitoring. Do not reduce observation frequency"
                f"\n        until the FSM confirms a return to Normal."
            )
        else:
            print(
                "\n  ✅ Vitals are Normal."
                "\n     All vital parameters are within their expected physiological ranges."
                "\n     No clinical concern flagged at this time."
            )
        return

    state_label = LABEL.get(curr_state, str(curr_state))
    print(f"\n  Current Condition: {state_label.upper()}")
    print("─" * 65)

    # ── Step A: Emergency-threshold vitals ────────────────
    print("\n  STEP A — EMERGENCY-THRESHOLD VITAL TRIGGERS")
    print("  (Vitals that have crossed the emergency boundary)")

    emergency_vitals = []
    for col, pfx in VITALS_MAP.items():
        val = row[col].values[0]
        h_key = f"{pfx}_high" if f"{pfx}_high" in TH else pfx
        l_key = f"{pfx}_low"  if f"{pfx}_low"  in TH else pfx

        hit_dir = None
        limit   = None
        level   = None

        allowed_dir = VITAL_DIRECTION.get(pfx, "both")  # 'low', 'high', or 'both'

        if allowed_dir in ("high", "both") and h_key in TH and val > TH[h_key][0]:
            hit_dir = "high"
            limits  = TH[h_key]
            level   = "Emergency" if val > limits[2] else ("Critical" if val > limits[1] else "Borderline-High")
            limit   = limits[0]
        elif allowed_dir in ("low", "both") and l_key in TH and val < TH[l_key][0]:
            hit_dir = "low"
            limits  = TH[l_key]
            level   = "Emergency" if val < limits[2] else ("Critical" if val < limits[1] else "Borderline-Low")
            limit   = limits[0]

        if hit_dir:
            emergency_vitals.append((col, val, hit_dir, limit, level))

    if emergency_vitals:
        vital_list = ", ".join(VITAL_NAMES.get(c, c) for c, *_ in emergency_vitals)
        print(
            f"\n  👉 Condition {state_label} is triggered due to abnormal vital(s):"
            f"\n     [{vital_list}]"
        )
        print()
        for col, val, direction, limit, level in emergency_vitals:
            name  = VITAL_NAMES.get(col, col)
            unit  = VITAL_UNITS.get(col, "")
            meaning = VITAL_CLINICAL_MEANING.get(col, {}).get(direction, "")
            side  = "above" if direction == "high" else "below"

            print(f"  🔴 {name}")
            print(f"       Current Value  : {val:.1f} {unit}")
            print(f"       Status         : {level} — {side} expected range of {limit} {unit}")
            if meaning:
                print(f"       Clinical Note  : This {meaning}.")
            print()
    else:
        print("\n  No individual vitals have breached normal thresholds.")

    # ── Step B: Active condition pattern flags ─────────────
    print("─" * 65)
    print("  STEP B — ACTIVE CLINICAL CONDITION PATTERNS")
    print("  (Conditions activated by multi-vital combinations)")

    found_any = False
    for flag, meta in CONDITIONS.items():
        if flag not in row.columns:
            continue
        if not (row[flag].values[0] == 1 or row[flag].values[0] == 1.0):
            continue

        found_any = True
        tier    = meta['tier']
        factors = CONDITION_FACTOR_SPECS[flag]
        cname   = CONDITION_NAMES.get(flag, flag)
        cdesc   = CONDITION_CLINICAL_DESC.get(flag, "")

        print(f"\n  🚨 {cname} DETECTED")
        print(f"     This condition is detected because of the combination of:")
        for f in factors:
            cur_val = row[f['raw']].values[0]
            fname   = VITAL_NAMES.get(f['raw'], f['raw'])
            funit   = VITAL_UNITS.get(f['raw'], "")
            side    = "above" if f['dir'] == 'high' else "below"
            print(
                f"       → {fname}: current {cur_val:.1f} {funit}"
                f" ({side} activation threshold of {f['thresh']} {funit},"
                f" normal reference: {f['normal_ref']} {funit})"
            )
        if cdesc:
            print(f"\n     📋 Clinical Context:")
            print(f"        {cdesc}")

    if not found_any:
        print("\n  No active multi-vital pattern conditions flagged at this time.")

    # ── Step C: All out-of-range vitals summary ────────────
    print()
    print("─" * 65)
    print("  STEP C — ALL VITALS NOT IN NORMAL RANGE")
    print("  (Including those that haven't crossed emergency level)")

    out_of_range = []
    for col, pfx in VITALS_MAP.items():
        val   = row[col].values[0]
        h_key = f"{pfx}_high" if f"{pfx}_high" in TH else pfx
        l_key = f"{pfx}_low"  if f"{pfx}_low"  in TH else pfx
        name  = VITAL_NAMES.get(col, col)
        unit  = VITAL_UNITS.get(col, "")

        allowed_dir = VITAL_DIRECTION.get(pfx, "both")  # 'low', 'high', or 'both'

        if allowed_dir in ("high", "both") and h_key in TH and val > TH[h_key][0]:
            limits = TH[h_key]
            level  = "Emergency" if val > limits[2] else ("Critical" if val > limits[1] else "Borderline-High")
            out_of_range.append(
                f"  ⚠️  {name}: {val:.1f} {unit}  →  HIGH — {level} range"
                f" (normal upper limit: {limits[0]} {unit})"
            )
        elif allowed_dir in ("low", "both") and l_key in TH and val < TH[l_key][0]:
            limits = TH[l_key]
            level  = "Emergency" if val < limits[2] else ("Critical" if val < limits[1] else "Borderline-Low")
            out_of_range.append(
                f"  ⚠️  {name}: {val:.1f} {unit}  →  LOW — {level} range"
                f" (normal lower limit: {limits[0]} {unit})"
            )

    if out_of_range:
        print()
        for line in out_of_range:
            print(line)
    else:
        print("\n  All individual vitals are within their defined normal range.")

stage_3(current_stat, curr_state, conf_state)


═════════════════════════════════════════════════════════════════
  STAGE 3 — VITAL SIGN ANALYSIS & CLINICAL PATTERN FLAGS
═════════════════════════════════════════════════════════════════

  Current Condition: CRITICAL
─────────────────────────────────────────────────────────────────

  STEP A — EMERGENCY-THRESHOLD VITAL TRIGGERS
  (Vitals that have crossed the emergency boundary)

  👉 Condition Critical is triggered due to abnormal vital(s):
     [Respiratory Rate, Diastolic Blood Pressure (DBP), End-Tidal CO₂ (EtCO₂)]

  🔴 Respiratory Rate
       Current Value  : 10.0 breaths/min
       Status         : Borderline-Low — below expected range of 12 breaths/min
       Clinical Note  : This means breathing is dangerously slow, which can lead to CO₂ retention and respiratory failure.

  🔴 Diastolic Blood Pressure (DBP)
       Current Value  : 55.0 mmHg
       Status         : Borderline-Low — below expected range of 60 mmHg
       Clinical Note  : This indicates low diastolic pressure, 

In [12]:
# ── STAGE 4: Trend & Noise Warnings ──────────────────────

def stage_4(row):
    VITALS = list(VITALS_MAP.keys())

    print("\n" + "═" * 65)
    print("  STAGE 4 — TREND & NOISE WARNINGS")
    print("═" * 65)

    # ── Warning 1: Monotonic slope trend in lag vitals ─────
    print("\n  WARNING 1 — CONTINUOUS VITAL TREND DETECTION")
    print("  (Checking lag vitals slope: 15m → 7m → 5m → 2m)")
    print()

    trend_found = False
    for v in VITALS:
        lag_col = f'lag_15m_{v}'
        # Only check trend warnings if this vital has a lag value present
        if lag_col not in row.columns:
            continue
        lag_val = row[lag_col].values[0]

        # Vitals that are not in normal range in lag — needed per spec
        pfx   = VITALS_MAP[v]
        h_key = f"{pfx}_high" if f"{pfx}_high" in TH else pfx
        l_key = f"{pfx}_low"  if f"{pfx}_low"  in TH else pfx
        lag_abnormal = False
        allowed_dir = VITAL_DIRECTION.get(pfx, "both")
        if allowed_dir in ("high", "both") and h_key in TH and lag_val > TH[h_key][0]:
            lag_abnormal = True
        elif allowed_dir in ("low", "both") and l_key in TH and lag_val < TH[l_key][0]:
            lag_abnormal = True

        if not lag_abnormal:
            continue  # only warn on vitals that were abnormal in lag

        s15 = row[f'slope_15m_{v}'].values[0]
        s7  = row[f'slope_7m_{v}'].values[0]
        s5  = row[f'slope_5m_{v}'].values[0]
        s2  = row[f'slope_2m_{v}'].values[0]

        name = VITAL_NAMES.get(v, v)
        unit = VITAL_UNITS.get(v, "")

        # Monotonically increasing: slope_15m < slope_7m < slope_5m < slope_2m
        if s15 < s7 < s5 < s2:
            direction_word = "increasing" if s2 > 0 else "accelerating upward in rate of change"
            print(
                f"  ⚠️  Warning — {name} state is changing continuously."
                f"\n     The slope is accelerating upward across all windows:"
                f"\n     15m slope={s15:+.4f}  →  7m slope={s7:+.4f}"
                f"  →  5m slope={s5:+.4f}  →  2m slope={s2:+.4f}"
                f"\n     This means {name} is {direction_word} and the rate is speeding up."
                f"\n     Current lag value: {lag_val:.1f} {unit}  (was abnormal 15m ago)"
            )
            print()
            trend_found = True

        # Monotonically decreasing: slope_15m > slope_7m > slope_5m > slope_2m
        elif s15 > s7 > s5 > s2:
            direction_word = "decreasing" if s2 < 0 else "decelerating in rate of change"
            print(
                f"  ⚠️  Warning — {name} state is changing continuously."
                f"\n     The slope is accelerating downward across all windows:"
                f"\n     15m slope={s15:+.4f}  →  7m slope={s7:+.4f}"
                f"  →  5m slope={s5:+.4f}  →  2m slope={s2:+.4f}"
                f"\n     This means {name} is {direction_word} and the rate is speeding up."
                f"\n     Current lag value: {lag_val:.1f} {unit}  (was abnormal 15m ago)"
            )
            print()
            trend_found = True

    if not trend_found:
        print("  ✅ No continuous monotonic trend detected across lag vitals.")

    # ── Warning 2: Impossible / noise values ───────────────
    print("─" * 65)
    print("  WARNING 2 — IMPOSSIBLE VITAL VALUES / SIGNAL NOISE")
    print("  (Checking for values beyond human physiological limits)")
    print()

    # Each entry: vital → (direction, impossible_limit, clinical_note)
    IMPOSSIBLE = {
        "pulse_pressure": ("low",  0.0,   "Pulse pressure of 0 mmHg is physiologically impossible in a living patient — the system may have a sensor disconnect or calculation error."),
        "spo2":           ("low",  20.0,  "SpO₂ below 20% is incompatible with life — this value is almost certainly a probe artefact or sensor error."),
        "heart_rate":     ("high", 250.0, "Heart rate above 250 bpm exceeds the physiological limit of the human conduction system — this is a sensor noise or data artefact."),
        "sbp":            ("low",  20.0,  "Systolic BP below 20 mmHg is not compatible with consciousness — this is likely a cuff error, arterial line artefact, or data gap."),
    }

    noise_found = False
    noise_vitals_hit = []

    for vital, (direction, limit, clinical_note) in IMPOSSIBLE.items():
        val = row[vital].values[0]
        hit = (direction == "low" and val <= limit) or (direction == "high" and val >= limit)
        name = VITAL_NAMES.get(vital, vital)
        unit = VITAL_UNITS.get(vital, "")

        if hit:
            print(
                f"  🚨 NOISE DETECTED — {name} = {val:.1f} {unit}"
                f"\n     Impossible human level: {direction} physiological limit = {limit} {unit}"
                f"\n     Clinical Note: {clinical_note}"
                f"\n     ⚡ Patient is flagged as Clinical High Emergency due to this signal."
            )
            print()
            noise_found = True
            noise_vitals_hit.append((vital, val, unit))

    # 80% proximity + high combined-score volatility check
    if row['pulse_pressure'].values[0] <= 5.0:  # near-zero pulse pressure
        pp_val = row['pulse_pressure'].values[0]
        for win in ['2m', '5m', '7m', '15m']:
            std_col = f'roll_std_{win}_combined'
            if std_col in row.columns:
                std_val = row[std_col].values[0]
                if std_val > 2.0:
                    print(
                        f"  🚨 COMBINED SCORE NOISE FLAG — High signal volatility detected."
                        f"\n     Pulse Pressure = {pp_val:.1f} mmHg (approaching impossible threshold of 0)."
                        f"\n     roll_std_{win}_combined = {std_val:.4f}  (abnormally high — indicates unstable measurements)."
                        f"\n     Over 80% of impossible vital proximity reached."
                        f"\n     Combined score instability at this level suggests signal noise is contaminating readings."
                        f"\n     ⚡ Recommend manual vital check before acting on these values."
                    )
                    noise_found = True
                    break

    if not noise_found:
        print("  ✅ No impossible vital values or signal noise artefacts detected.")

stage_4(current_stat)


═════════════════════════════════════════════════════════════════
  STAGE 4 — TREND & NOISE WARNINGS
═════════════════════════════════════════════════════════════════

  WARNING 1 — CONTINUOUS VITAL TREND DETECTION
  (Checking lag vitals slope: 15m → 7m → 5m → 2m)

  ✅ No continuous monotonic trend detected across lag vitals.
─────────────────────────────────────────────────────────────────
  WARNING 2 — IMPOSSIBLE VITAL VALUES / SIGNAL NOISE
  (Checking for values beyond human physiological limits)

  ✅ No impossible vital values or signal noise artefacts detected.


In [13]:
# ── STAGE 5: Good Signs / Recovery Detection ─────────────

def stage_5(row, past_row):
    print("\n" + "═" * 65)
    print("  STAGE 5 — GOOD SIGNS & RECOVERY ANALYSIS")
    print("═" * 65)

    if past_row is None:
        print(
            "\n  ℹ️  No 15-minute lag snapshot is available for this patient at this time."
            "\n     Recovery analysis requires prior readings to compare against."
            "\n     Skipping recovery check — ensure lag data is recorded for future assessments."
        )
        return

    found_any = False

    for col, pfx in VITALS_MAP.items():
        lag_col = f'lag_15m_{col}'
        if lag_col in row.columns:
            past_val = row[lag_col].values[0]
        else:
            past_val = past_row[col].values[0]

        curr_val = row[col].values[0]
        name     = VITAL_NAMES.get(col, col)
        unit     = VITAL_UNITS.get(col, "")

        s15 = row[f'slope_15m_{col}'].values[0]
        s7  = row[f'slope_7m_{col}'].values[0]
        s5  = row[f'slope_5m_{col}'].values[0]
        s2  = row[f'slope_2m_{col}'].values[0]

        all_neg = s15 < 0 and s7 < 0 and s5 < 0 and s2 < 0
        all_pos = s15 > 0 and s7 > 0 and s5 > 0 and s2 > 0

        h_key = f"{pfx}_high" if f"{pfx}_high" in TH else pfx
        l_key = f"{pfx}_low"  if f"{pfx}_low"  in TH else pfx

        # Was HIGH-abnormal 15m ago AND all slopes now negative (coming down — good)
        if h_key in TH and past_val > TH[h_key][0] and all_neg:
            normal_limit = TH[h_key][0]
            print(
                f"  ✅ Good Sign — Condition changing over time:"
                f"\n     {name} is DECREASING."
                f"\n     15 minutes ago this vital was {past_val:.1f} {unit}, which was ABOVE the"
                f"\n     normal upper limit of {normal_limit} {unit} — indicating an elevated state."
                f"\n     Current value: {curr_val:.1f} {unit}."
                f"\n     All four slope windows confirm a consistent downward movement:"
                f"\n     15m={s15:+.4f}  7m={s7:+.4f}  5m={s5:+.4f}  2m={s2:+.4f}"
                f"\n     This is a positive sign — the vital is trending back toward normal range."
            )
            print()
            found_any = True

        # Was LOW-abnormal 15m ago AND all slopes now positive (coming up — good)
        elif l_key in TH and past_val < TH[l_key][0] and all_pos:
            normal_limit = TH[l_key][0]
            print(
                f"  ✅ Good Sign — Condition changing over time:"
                f"\n     {name} is INCREASING."
                f"\n     15 minutes ago this vital was {past_val:.1f} {unit}, which was BELOW the"
                f"\n     normal lower limit of {normal_limit} {unit} — indicating a deficient state."
                f"\n     Current value: {curr_val:.1f} {unit}."
                f"\n     All four slope windows confirm a consistent upward movement:"
                f"\n     15m={s15:+.4f}  7m={s7:+.4f}  5m={s5:+.4f}  2m={s2:+.4f}"
                f"\n     This is a positive sign — the vital is trending back toward normal range."
            )
            print()
            found_any = True

    if not found_any:
        print(
            "\n  ℹ️  No recovery signs detected at this time."
            "\n     No vitals that were previously abnormal are showing a consistent"
            "\n     return trend across all slope windows."
        )

stage_5(current_stat, past_stat)


═════════════════════════════════════════════════════════════════
  STAGE 5 — GOOD SIGNS & RECOVERY ANALYSIS
═════════════════════════════════════════════════════════════════
  ✅ Good Sign — Condition changing over time:
     Heart Rate is INCREASING.
     15 minutes ago this vital was 57.0 bpm, which was BELOW the
     normal lower limit of 60 bpm — indicating a deficient state.
     Current value: 83.0 bpm.
     All four slope windows confirm a consistent upward movement:
     15m=+0.0688  7m=+0.1195  5m=+0.1154  2m=+0.0205
     This is a positive sign — the vital is trending back toward normal range.



In [14]:
# ── FINAL SUMMARY REPORT ─────────────────────────────────

def final_report(row, past_row):
    LABEL  = {0: "NORMAL", 1: "CRITICAL", 2: "EMERGENCY"}
    curr   = row['severity_label'].values[0]
    conf   = row['result_label'].values[0]
    score  = row['combined_score'].values[0]
    future = row['future_label'].values[0] if 'future_label' in row.columns else 'N/A'

    # Active flags
    active_flags = [
        f for f in CONDITION_FACTOR_SPECS
        if f in row.columns and (row[f].values[0] == 1 or row[f].values[0] == 1.0)
    ]

    # Trend warnings
    trend_up, trend_down = [], []
    for v in VITALS_MAP.keys():
        lag_col = f'lag_15m_{v}'
        if lag_col not in row.columns:
            continue
        lag_val = row[lag_col].values[0]
        pfx = VITALS_MAP[v]
        h_key = f"{pfx}_high" if f"{pfx}_high" in TH else pfx
        l_key = f"{pfx}_low"  if f"{pfx}_low"  in TH else pfx
        lag_abnormal = (h_key in TH and lag_val > TH[h_key][0]) or \
                       (l_key in TH and lag_val < TH[l_key][0])
        if not lag_abnormal:
            continue
        s = [row[f'slope_{w}m_{v}'].values[0] for w in [15, 7, 5, 2]]
        if s[0] < s[1] < s[2] < s[3]:
            trend_up.append(VITAL_NAMES.get(v, v))
        elif s[0] > s[1] > s[2] > s[3]:
            trend_down.append(VITAL_NAMES.get(v, v))

    # Recovery signs
    recovery = []
    if past_row is not None:
        for col, pfx in VITALS_MAP.items():
            lag_col = f'lag_15m_{col}'
            pv = row[lag_col].values[0] if lag_col in row.columns else past_row[col].values[0]
            s  = [row[f'slope_{w}m_{col}'].values[0] for w in [15, 7, 5, 2]]
            h_key = f"{pfx}_high" if f"{pfx}_high" in TH else pfx
            l_key = f"{pfx}_low"  if f"{pfx}_low"  in TH else pfx
            if VITAL_DIRECTION.get(pfx, "both") in ("high", "both") and h_key in TH and pv > TH[h_key][0] and all(x < 0 for x in s):
                recovery.append(VITAL_NAMES.get(col, col))
            elif VITAL_DIRECTION.get(pfx, "both") in ("low", "both") and l_key in TH and pv < TH[l_key][0] and all(x > 0 for x in s):
                recovery.append(VITAL_NAMES.get(col, col))

    SEP = "═" * 65
    print("\n" + SEP)
    print("         RULE-BASED EXPLAINABLE AI LAYER — FINAL REPORT")
    print(SEP)
    print(f"  Patient ID         : {TARGET_PATIENT_ID}")
    print(f"  Time               : {TARGET_TIME}s  ({TARGET_TIME // 60} min into monitoring)")
    print(f"  Combined Score     : {score:.4f}")
    print("─" * 65)
    print(f"  Raw Severity       : {LABEL.get(curr, curr)}")
    print(f"  FSM Confirmed      : {LABEL.get(conf, conf)}")
    future_label = LABEL.get(future, str(future)) if isinstance(future, int) else str(future)
    print(f"  15-Min Forecast    : {future_label}")
    print("─" * 65)

    # Overall status
    if curr == 0 and conf == 0:
        print("  ✅  PATIENT STABLE — All vitals within normal range.")
        print("      No deterioration pattern detected. Continue routine monitoring.")
    elif curr == 0 and conf != 0:
        print(f"  ⚠️   WATCH STATE — Vitals currently normal.")
        print(f"      FSM still holds {LABEL.get(conf, conf)} due to prior deterioration window.")
        print(f"      Do not reduce monitoring. Await FSM confirmation of Normal.")
    elif curr != conf:
        print(f"  ⚠️   TRANSITIONING — Raw: {LABEL.get(curr, curr)} | FSM: {LABEL.get(conf, conf)}")
        print(f"      System is in a state transition window. Close observation required.")
    else:
        print(f"  🚨  CONFIRMED {LABEL.get(conf, conf)} — Raw severity and FSM both agree.")
        print(f"      Sustained deterioration confirmed. medical action required.")

    print()

    if active_flags:
        tier_labels = [f"{CONDITION_NAMES.get(f, f)} (Tier {CONDITIONS[f]['tier']})" for f in active_flags]
        print(f"  ⚡ Active Condition Flags ({len(active_flags)}):")
        for t in tier_labels:
            print(f"      → {t}")
    else:
        print("  ✅ Active Flags       : None")

    print()
    if trend_up:
        print(f"  ⚠️  Continuously Rising Trend  : {', '.join(trend_up)}")
    if trend_down:
        print(f"  ⚠️  Continuously Falling Trend : {', '.join(trend_down)}")
    if not trend_up and not trend_down:
        print("  ✅ Trends             : No monotonic trends detected in abnormal lag vitals")

    print()
    if recovery:
        print(f"  ✅ Recovery Signs     : {', '.join(recovery)}")
        print(f"     These vitals were previously abnormal and are now trending toward normal.")
    else:
        print("  ─  Recovery Signs     : None detected at this time")

    print(SEP)

final_report(current_stat, past_stat)


═════════════════════════════════════════════════════════════════
         RULE-BASED EXPLAINABLE AI LAYER — FINAL REPORT
═════════════════════════════════════════════════════════════════
  Patient ID         : 64
  Time               : 3600s  (60 min into monitoring)
  Combined Score     : 1.2054
─────────────────────────────────────────────────────────────────
  Raw Severity       : CRITICAL
  FSM Confirmed      : CRITICAL
  15-Min Forecast    : 2.0
─────────────────────────────────────────────────────────────────
  🚨  CONFIRMED CRITICAL — Raw severity and FSM both agree.
      Sustained deterioration confirmed. Immediate clinical attention required.

  ✅ Active Flags       : None

  ✅ Trends             : No monotonic trends detected in abnormal lag vitals

  ✅ Recovery Signs     : Heart Rate
     These vitals were previously abnormal and are now trending toward normal.
═════════════════════════════════════════════════════════════════
